In [1]:
# Read panel data with merged macro + AQI features.
import pandas as pd
pd.set_option('display.max_columns', None)

from pathlib import Path
import pandas as pd

ROOT = Path('/Users/jinyecai/Desktop/ML_Mortgage')
PROC = ROOT / 'processed_mortgage'
MACRO_MERGED = PROC / 'macro_merged'

default_path = MACRO_MERGED / 'panel_default_with_macro.parquet'
prepay_path = MACRO_MERGED / 'panel_prepay_with_macro.parquet'

default_data = pd.read_parquet(default_path)
prepay_data = pd.read_parquet(prepay_path)

print(f'default_data: {default_data.shape}')
print(f'prepay_data:  {prepay_data.shape}')
print('default monthly range:', default_data['monthly_reporting_period'].min(), '->', default_data['monthly_reporting_period'].max())
print('prepay monthly range: ', prepay_data['monthly_reporting_period'].min(), '->', prepay_data['monthly_reporting_period'].max())

default_data.head()


default_data: (1785162, 59)
prepay_data:  (1785162, 59)
default monthly range: 201601 -> 202509
prepay monthly range:  201601 -> 202509


,loan_sequence_number,monthly_reporting_period,rp_yyyymm,file_year_x,loan_id,loan_age,remaining_months_to_legal_maturity,current_actual_upb,interest_bearing_upb,current_interest_rate,estimated_loan_to_value_eltv,modification_flag,y_default,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,metropolitan_statistical_area_msa_or_metropolitan_division,mortgage_insurance_percentage_mi_pct,number_of_units,occupancy_status,original_combined_loan_to_value_cltv,original_debt_to_income_dti_ratio,original_upb,original_loan_to_value_ltv,original_interest_rate,channel,prepayment_penalty_mortgage_ppm_flag,amortization_type_formerly_product_type,property_state,property_type,postal_code,loan_purpose,original_loan_term,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,harp_indicator,property_valuation_method,interest_only_i_o_indicator,mortgage_insurance_cancellation_indicator,file_year_y,first_payment_date_dt,maturity_date_dt,year,unrate,cpi,fedfunds,gs10,cpi_yoy,pmms30,fhfa_hpi,fhfa_hpi_yoy,median_aqi,mortgage_treasury_spread,spread_at_origination
0,F16Q10000149,201602,201602,2016,F16Q10000149,000,180,60000.00,60000.00,3.25,,,0,671,201603,N,203102,<NA>,000,1,P,34,42,60000,34,3.25,R,N,FRM,IL,SF,62400,C,180,1,Other sellers,Other servicers,<NA>,<NA>,9,<NA>,7,N,7,2016,2016-03-01,2031-02-01,2016,4.9,237.336,0.38,1.78,0.008473,3.660,177.94,0.026419,42.0,1.880,-0.41
1,F16Q10000149,201603,201603,2016,F16Q10000149,001,179,60000.00,60000.00,3.25,,,0,671,201603,N,203102,<NA>,000,1,P,34,42,60000,34,3.25,R,N,FRM,IL,SF,62400,C,180,1,Other sellers,Other servicers,<NA>,<NA>,9,<NA>,7,N,7,2016,2016-03-01,2031-02-01,2016,5.0,238.080,0.36,1.89,0.008916,3.694,177.94,0.026419,42.0,1.804,-0.41
2,F16Q10000149,201604,201604,2016,F16Q10000149,002,178,59000.00,59000.00,3.25,,,0,671,201603,N,203102,<NA>,000,1,P,34,42,60000,34,3.25,R,N,FRM,IL,SF,62400,C,180,1,Other sellers,Other servicers,<NA>,<NA>,9,<NA>,7,N,7,2016,2016-03-01,2031-02-01,2016,5.1,238.992,0.37,1.81,0.011726,3.605,186.31,0.023850,42.0,1.795,-0.41
3,F16Q10000149,201605,201605,2016,F16Q10000149,003,177,59000.00,59000.00,3.25,,,0,671,201603,N,203102,<NA>,000,1,P,34,42,60000,34,3.25,R,N,FRM,IL,SF,62400,C,180,1,Other sellers,Other servicers,<NA>,<NA>,9,<NA>,7,N,7,2016,2016-03-01,2031-02-01,2016,4.8,239.557,0.37,1.81,0.010785,3.600,186.31,0.023850,42.0,1.790,-0.41
4,F16Q10000149,201606,201606,2016,F16Q10000149,004,176,59000.00,59000.00,3.25,,,0,671,201603,N,203102,<NA>,000,1,P,34,42,60000,34,3.25,R,N,FRM,IL,SF,62400,C,180,1,Other sellers,Other servicers,<NA>,<NA>,9,<NA>,7,N,7,2016,2016-03-01,2031-02-01,2016,4.9,240.222,0.38,1.64,0.010793,3.568,186.31,0.023850,42.0,1.928,-0.41


In [5]:
import pandas as pd
import numpy as np

CATEGORICAL_COLS = [
    'vintage_group',
    'number_of_units',
    'occupancy_status',
    'channel',
    'state_region',
    'property_type',
    'loan_purpose',
    'super_conforming_flag',
    'program_indicator',
    'harp_indicator',
    'interest_only_i_o_indicator',
    'mortgage_insurance_cancellation_indicator',
]

DROP_COLS = [
    'loan_sequence_number',
    'monthly_reporting_period',
    'file_year_y',
    'remaining_months_to_legal_maturity',
    'pre_harp_loan_sequence_number',
    'property_valuation_method',
    'interest_bearing_upb',
    # 'mortgage_treasury_spread',   
    # 'current_interest_rate',
    'fhfa_hpi',
    'modification_flag',
]

RENAME_COLS = {
    'original_combined_loan_to_value_cltv': 'origcltv',
    'original_debt_to_income_dti_ratio': 'dti',
    'metropolitan_statistical_area_msa_or_metropolitan_division': 'msa_code',
}

STATE_REGION_MAP = {
    'CT': 'NE', 'ME': 'NE', 'MA': 'NE', 'NH': 'NE', 'RI': 'NE', 'VT': 'NE',
    'NJ': 'NE', 'NY': 'NE', 'PA': 'NE',
    'IL': 'MW', 'IN': 'MW', 'MI': 'MW', 'OH': 'MW', 'WI': 'MW',
    'IA': 'MW', 'KS': 'MW', 'MN': 'MW', 'MO': 'MW', 'NE': 'MW', 'ND': 'MW', 'SD': 'MW',
    'DE': 'SE', 'DC': 'SE', 'FL': 'SE', 'GA': 'SE', 'MD': 'SE', 'NC': 'SE', 'SC': 'SE',
    'VA': 'SE', 'WV': 'SE', 'AL': 'SE', 'KY': 'SE', 'MS': 'SE', 'TN': 'SE',
    'AR': 'SE', 'LA': 'SE',
    'AZ': 'SW', 'NM': 'SW', 'OK': 'SW', 'TX': 'SW',
    'AK': 'W', 'CA': 'W', 'CO': 'W', 'HI': 'W', 'ID': 'W', 'MT': 'W', 'NV': 'W',
    'OR': 'W', 'UT': 'W', 'WA': 'W', 'WY': 'W',
    'PR': 'OT', 'VI': 'OT', 'GU': 'OT',
}

BASELINE_EXCLUDE_COLS = {
    'loan_id',
    'rp_yyyymm',
    'first_payment_date',
    'maturity_date',
    'first_payment_date_dt',
    'maturity_date_dt',
    'postal_code',
    'msa_code',
    'seller_name',
    'servicer_name',
    'year',
}

def map_vintage_group(series: pd.Series) -> pd.Series:
    years = pd.to_numeric(series, errors='coerce')
    groups = pd.Series('missing', index=series.index, dtype='string')
    groups = groups.mask(years.le(2019), 'pre_covid_vintages')
    groups = groups.mask(years.between(2020, 2022, inclusive='both'), 'covid_vintages')
    groups = groups.mask(years.ge(2023), 'post_covid_vintages')
    return groups

def coerce_numeric_like(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        series = df[col]
        if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
            continue
        cleaned = series.astype('string').replace({'': pd.NA, '<NA>': pd.NA})
        converted = pd.to_numeric(cleaned, errors='coerce')
        if cleaned.notna().sum() == converted.notna().sum():
            df[col] = converted
        else:
            df[col] = cleaned
    return df

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'file_year_x' in df.columns:
        df = df.rename(columns={'file_year_x': 'vintage_year'})

    if 'vintage_year' in df.columns:
        df['vintage_group'] = map_vintage_group(df['vintage_year'])
        df = df.drop(columns=['vintage_year'])

    rename_map = {old: new for old, new in RENAME_COLS.items() if old in df.columns}
    if rename_map:
        df = df.rename(columns=rename_map)

    sort_cols = [col for col in ['loan_id', 'monthly_reporting_period'] if col in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).copy()

    if 'first_time_homebuyer_flag' in df.columns:
        fthb = df['first_time_homebuyer_flag'].astype('string').str.strip().str.upper()
        fthb = fthb.fillna('').replace('', 'N').replace('9', 'N')
        df['first_time_homebuyer_flag'] = fthb.eq('Y').astype('int8')

    if 'number_of_borrowers' in df.columns:
        borrowers = pd.to_numeric(df['number_of_borrowers'], errors='coerce')
        df['number_of_borrowers'] = borrowers.gt(1).astype('int8')

    if 'property_state' in df.columns:
        state = df['property_state'].astype('string').str.strip().str.upper()
        state = state.fillna('').replace('', 'missing').replace('<NA>', 'missing')
        df['state_region'] = state.map(STATE_REGION_MAP).fillna('OT')
        df = df.drop(columns=['property_state'])

    # drop no variation columns
    for col in ['prepayment_penalty_mortgage_ppm_flag', 'amortization_type_formerly_product_type']:
        if col in df.columns:
            df = df.drop(columns=[col])

    # binary categorical convert to 0/1；leave multiclass sklearn one-hot
    binary_yn_cols = [
        'super_conforming_flag',
        'program_indicator',
        'harp_indicator',
        'interest_only_i_o_indicator',
        'mortgage_insurance_cancellation_indicator',
    ]
    for col in binary_yn_cols:
        if col in df.columns:
            s = df[col].astype('string').str.strip().str.upper().fillna('N')
            df[col] = s.eq('Y').astype('int8')

    
    for col in CATEGORICAL_COLS:
        if col in df.columns and col not in binary_yn_cols:
            df[col] = (
                df[col]
                .astype('string')
                .str.strip()
                .fillna('missing')
                .replace('', 'missing')
            )

    drop_cols = [col for col in DROP_COLS if col in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df = coerce_numeric_like(df)
    return df

default_fe = feature_engineering(default_data)
prepay_fe = feature_engineering(prepay_data)

In [9]:
print('default_fe shape:', default_fe.shape)

default_fe shape: (1785162, 48)


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder

BASELINE_EXCLUDE_COLS = {
    'loan_id',
    'rp_yyyymm',
    'first_payment_date',
    'maturity_date',
    'first_payment_date_dt',
    'maturity_date_dt',
    'postal_code',
    'msa_code',
    'seller_name',
    'servicer_name',
    'year'
}

def coerce_numeric_like(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        series = df[col]
        if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
            continue
        cleaned = series.astype('string').replace({'': pd.NA, '<NA>': pd.NA})
        converted = pd.to_numeric(cleaned, errors='coerce')
        if cleaned.notna().sum() == converted.notna().sum():
            df[col] = converted
        else:
            df[col] = cleaned
    return df

def prepare_baseline_frame(df: pd.DataFrame, target_col: str, already_engineered: bool = False,
                           feature_cols: list[str] | None = None):
    fe = df.copy() if already_engineered else feature_engineering(df)
    loan_ids = fe['loan_id'].copy()
    y = fe[target_col].astype(int).copy()
    X = fe.drop(columns=[target_col]).copy()

    drop_cols = [col for col in BASELINE_EXCLUDE_COLS if col in X.columns]
    X = X.drop(columns=drop_cols)

    X = coerce_numeric_like(X)

    if feature_cols is not None:
        missing_feature_cols = [col for col in feature_cols if col not in X.columns]
        if missing_feature_cols:
            raise KeyError(f"Requested feature_cols not found in baseline frame: {missing_feature_cols[:10]}")
        X = X.loc[:, feature_cols].copy()

    return X, y, loan_ids

def run_baseline_model(df: pd.DataFrame, target_col: str, label: str,
                       test_size: float = 0.2, random_state: int = 42,
                       already_engineered: bool = False,
                       feature_cols: list[str] | None = None):
    X, y, loan_ids = prepare_baseline_frame(
        df,
        target_col,
        already_engineered=already_engineered,
        feature_cols=feature_cols,
    )

    loan_level_target = pd.DataFrame({
        'loan_id': loan_ids,
        target_col: y
    }).groupby('loan_id')[target_col].max()

    train_loans, test_loans = train_test_split(
        loan_level_target.index,
        test_size=test_size,
        random_state=random_state,
        stratify=loan_level_target.values,
    )

    train_mask = loan_ids.isin(train_loans).to_numpy()
    test_mask = loan_ids.isin(test_loans).to_numpy()

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]
    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]

    categorical_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
            ]), categorical_cols),
        ],
        remainder='drop',
    )

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('scaler', MaxAbsScaler()),
        ('classifier', SGDClassifier(
            loss='log_loss',
            penalty='l2',
            alpha=0.0001,
            class_weight=None,
            max_iter=1000,
            tol=1e-3,
            random_state=random_state,
        )),
    ])

    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:, 1]
    y_score_clipped = np.clip(y_score, 1e-12, 1 - 1e-12)

    feature_names = model.named_steps['preprocessor'].get_feature_names_out()

    return {
        'model': label,
        'train_rows': int(train_mask.sum()),
        'test_rows': int(test_mask.sum()),
        'n_features_raw': int(X.shape[1]),
        'n_features_encoded': int(len(feature_names)),
        'test_event_rate': float(y_test.mean()),
        'roc_auc': float(roc_auc_score(y_test, y_score)),
        'pr_auc': float(average_precision_score(y_test, y_score)),
        'log_loss': float(log_loss(y_test, y_score_clipped, labels=[0, 1])),
        'brier_score': float(brier_score_loss(y_test, y_score)),
    }


baseline_results = pd.DataFrame([
    run_baseline_model(default_fe, 'y_default', 'default_baseline', already_engineered=True),
    run_baseline_model(prepay_fe, 'y_prepay', 'prepay_baseline', already_engineered=True),
]).round(4)

baseline_results

,model,train_rows,test_rows,n_features_raw,n_features_encoded,test_event_rate,roc_auc,pr_auc,log_loss,brier_score
0,default_baseline,1424249,360913,36,47,0.0008,0.7837,0.0049,0.0061,0.0008
1,prepay_baseline,1430718,354444,36,47,0.0108,0.9845,0.7377,0.0226,0.0057


In [14]:
prepay_fe.head()

,rp_yyyymm,loan_id,loan_age,current_actual_upb,current_interest_rate,estimated_loan_to_value_eltv,y_prepay,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa_code,mortgage_insurance_percentage_mi_pct,number_of_units,occupancy_status,origcltv,dti,original_upb,original_loan_to_value_ltv,original_interest_rate,channel,property_type,postal_code,loan_purpose,original_loan_term,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,program_indicator,harp_indicator,interest_only_i_o_indicator,mortgage_insurance_cancellation_indicator,first_payment_date_dt,maturity_date_dt,year,unrate,cpi,fedfunds,gs10,cpi_yoy,pmms30,fhfa_hpi_yoy,median_aqi,mortgage_treasury_spread,incentive_rate,vintage_group,state_region
0,201602,F16Q10000149,0,60000.0,3.25,<NA>,0,671,201603,0,203102,<NA>,0,1,P,34,42,60000,34,3.25,R,SF,62400,C,180,0,Other sellers,Other servicers,0,0,0,0,0,2016-03-01,2031-02-01,2016,4.9,237.336,0.38,1.78,0.008473,3.660,0.026419,42.0,1.880,-0.41,pre_covid_vintages,MW
1,201603,F16Q10000149,1,60000.0,3.25,<NA>,0,671,201603,0,203102,<NA>,0,1,P,34,42,60000,34,3.25,R,SF,62400,C,180,0,Other sellers,Other servicers,0,0,0,0,0,2016-03-01,2031-02-01,2016,5.0,238.080,0.36,1.89,0.008916,3.694,0.026419,42.0,1.804,-0.444,pre_covid_vintages,MW
2,201604,F16Q10000149,2,59000.0,3.25,<NA>,0,671,201603,0,203102,<NA>,0,1,P,34,42,60000,34,3.25,R,SF,62400,C,180,0,Other sellers,Other servicers,0,0,0,0,0,2016-03-01,2031-02-01,2016,5.1,238.992,0.37,1.81,0.011726,3.605,0.023850,42.0,1.795,-0.355,pre_covid_vintages,MW
3,201605,F16Q10000149,3,59000.0,3.25,<NA>,0,671,201603,0,203102,<NA>,0,1,P,34,42,60000,34,3.25,R,SF,62400,C,180,0,Other sellers,Other servicers,0,0,0,0,0,2016-03-01,2031-02-01,2016,4.8,239.557,0.37,1.81,0.010785,3.600,0.023850,42.0,1.790,-0.35,pre_covid_vintages,MW
4,201606,F16Q10000149,4,59000.0,3.25,<NA>,0,671,201603,0,203102,<NA>,0,1,P,34,42,60000,34,3.25,R,SF,62400,C,180,0,Other sellers,Other servicers,0,0,0,0,0,2016-03-01,2031-02-01,2016,4.9,240.222,0.38,1.64,0.010793,3.568,0.023850,42.0,1.928,-0.318,pre_covid_vintages,MW


In [18]:
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder


def evaluate_binary_predictions(y_true, y_score) -> dict:
    y_score_clipped = np.clip(y_score, 1e-12, 1 - 1e-12)
    return {
        'test_event_rate': float(np.mean(y_true)),
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'pr_auc': float(average_precision_score(y_true, y_score)),
        'log_loss': float(log_loss(y_true, y_score_clipped, labels=[0, 1])),
        'brier_score': float(brier_score_loss(y_true, y_score)),
    }


def build_train_test_split(df: pd.DataFrame,
                           target_col: str,
                           already_engineered: bool = False,
                           feature_cols: list[str] | None = None,
                           test_size: float = 0.2,
                           random_state: int = 42):
    X, y, loan_ids = prepare_baseline_frame(
        df,
        target_col,
        already_engineered=already_engineered,
        feature_cols=feature_cols,
    )

    loan_level_target = pd.DataFrame({
        'loan_id': loan_ids,
        target_col: y
    }).groupby('loan_id')[target_col].max()

    train_loans, test_loans = train_test_split(
        loan_level_target.index,
        test_size=test_size,
        random_state=random_state,
        stratify=loan_level_target.values,
    )

    train_mask = loan_ids.isin(train_loans).to_numpy()
    test_mask = loan_ids.isin(test_loans).to_numpy()

    X_train = X.loc[train_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[train_mask].copy()
    y_test = y.loc[test_mask].copy()

    return X_train, X_test, y_train, y_test


def build_preprocessor_from_train(X_train: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
            ]), categorical_cols),
        ],
        remainder='drop',
    )
    return preprocessor


def map_encoded_feature_to_source(encoded_name: str, raw_categorical_cols: list[str]) -> str:
    if encoded_name.startswith('num__'):
        return encoded_name.split('num__', 1)[1]

    if encoded_name.startswith('cat__'):
        raw = encoded_name.split('cat__', 1)[1]
        matches = [col for col in raw_categorical_cols if raw == col or raw.startswith(f'{col}_')]
        if matches:
            return max(matches, key=len)
        return raw

    return encoded_name


def run_regularized_feature_selection(
    df: pd.DataFrame,
    target_col: str,
    label: str,
    already_engineered: bool = False,
    feature_cols: list[str] | None = None,
    test_size: float = 0.2,
    random_state: int = 42,
    screening_C: float = 0.05,
    final_alpha: float = 0.0001,
):
    # 1) train/test split
    X_train, X_test, y_train, y_test = build_train_test_split(
        df,
        target_col,
        already_engineered=already_engineered,
        feature_cols=feature_cols,
        test_size=test_size,
        random_state=random_state,
    )

    raw_categorical_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

    # 2) preprocess + scale
    preprocessor = build_preprocessor_from_train(X_train)

    X_train_enc = preprocessor.fit_transform(X_train)
    X_test_enc = preprocessor.transform(X_test)
    encoded_feature_names = preprocessor.get_feature_names_out()

    scaler = MaxAbsScaler()
    X_train_scaled = scaler.fit_transform(X_train_enc)
    X_test_scaled = scaler.transform(X_test_enc)

    # 3) L1 screening
    screening_model = LogisticRegression(
        penalty='l1',
        C=screening_C,
        solver='saga',
        max_iter=2000,
        random_state=random_state,
        n_jobs=-1,
    )
    screening_model.fit(X_train_scaled, y_train)

    screening_coef = screening_model.coef_.ravel()
    nonzero_mask = screening_coef != 0

    if nonzero_mask.sum() == 0:
        screening_model = LogisticRegression(
            penalty='l1',
            C=0.2,
            solver='saga',
            max_iter=2000,
            random_state=random_state,
            n_jobs=-1,
        )
        screening_model.fit(X_train_scaled, y_train)
        screening_coef = screening_model.coef_.ravel()
        nonzero_mask = screening_coef != 0

    selected_encoded_features = encoded_feature_names[nonzero_mask].tolist()

    if len(selected_encoded_features) == 0:
        raise ValueError("L1 screening selected zero features even after relaxing C. Try a larger C.")

    # 4) final model on selected encoded features
    X_train_final = X_train_scaled[:, nonzero_mask]
    X_test_final = X_test_scaled[:, nonzero_mask]

    final_model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=final_alpha,
        class_weight=None,
        max_iter=1000,
        tol=1e-3,
        random_state=random_state,
    )
    final_model.fit(X_train_final, y_train)

    final_score = final_model.predict_proba(X_test_final)[:, 1]
    final_metrics = evaluate_binary_predictions(y_test, final_score)

    final_metrics_table = pd.DataFrame([
        {
            'model': f'{label}_final_after_l1_screening',
            'train_rows': int(X_train.shape[0]),
            'test_rows': int(X_test.shape[0]),
            'n_features_raw': int(X_train.shape[1]),
            'n_features_encoded_all': int(len(encoded_feature_names)),
            'n_selected_encoded_features': int(len(selected_encoded_features)),
            **final_metrics,
        }
    ]).round(6)

    # 5) coefficient tables
    selected_coefs = final_model.coef_.ravel()

    encoded_coef_table = pd.DataFrame({
        'encoded_feature': selected_encoded_features,
        'coefficient': selected_coefs,
    })
    encoded_coef_table['abs_coefficient'] = encoded_coef_table['coefficient'].abs()
    encoded_coef_table['odds_ratio'] = np.exp(encoded_coef_table['coefficient'])
    encoded_coef_table['source_feature'] = encoded_coef_table['encoded_feature'].map(
        lambda x: map_encoded_feature_to_source(x, raw_categorical_cols)
    )
    encoded_coef_table = encoded_coef_table.sort_values(
        ['abs_coefficient', 'encoded_feature'],
        ascending=[False, True]
    ).reset_index(drop=True)

    source_feature_summary = (
        encoded_coef_table
        .groupby('source_feature', as_index=False)
        .agg(
            n_selected_components=('encoded_feature', 'size'),
            max_abs_coefficient=('abs_coefficient', 'max'),
            mean_abs_coefficient=('abs_coefficient', 'mean'),
        )
        .sort_values(['max_abs_coefficient', 'n_selected_components'], ascending=[False, False])
        .reset_index(drop=True)
    )

    return {
        'final_metrics_table': final_metrics_table,
        'final_metrics': final_metrics,
        'preprocessor': preprocessor,
        'scaler': scaler,
        'screening_model': screening_model,
        'final_model': final_model,
        'encoded_feature_names_all': encoded_feature_names.tolist(),
        'selected_encoded_features': selected_encoded_features,
        'encoded_coef_table': encoded_coef_table,
        'source_feature_summary': source_feature_summary,
        'X_train_shape': X_train.shape,
        'X_test_shape': X_test.shape,
    }

In [19]:
prepay_regularized = run_regularized_feature_selection(
    prepay_fe,
    target_col='y_prepay',
    label='prepay',
    already_engineered=True,
    screening_C=0.05,  
    final_alpha=0.0001,
)

prepay_regularized['final_metrics_table']
prepay_regularized['source_feature_summary']
prepay_regularized['encoded_coef_table']

,encoded_feature,coefficient,abs_coefficient,odds_ratio,source_feature
0,num__current_actual_upb,-7.900324,7.900324,0.000371,current_actual_upb
1,num__estimated_loan_to_value_eltv,4.909597,4.909597,135.584743,estimated_loan_to_value_eltv
2,num__original_upb,2.008053,2.008053,7.448804,original_upb
3,num__loan_age,1.670436,1.670436,5.314483,loan_age
4,num__gs10,-1.614093,1.614093,0.199071,gs10
5,cat__property_type_PU,1.365399,1.365399,3.917286,property_type
6,cat__property_type_SF,1.257259,1.257259,3.515772,property_type
7,cat__property_type_MH,-0.865049,0.865049,0.421031,property_type
8,cat__occupancy_status_P,0.748730,0.748730,2.114313,occupancy_status
9,num__mortgage_insurance_cancellation_indicator,-0.658125,0.658125,0.517821,mortgage_insurance_cancellation_indicator


In [21]:
prepay_regularized['final_metrics_table']

,model,train_rows,test_rows,n_features_raw,n_features_encoded_all,n_selected_encoded_features,test_event_rate,roc_auc,pr_auc,log_loss,brier_score
0,prepay_final_after_l1_screening,1430718,354444,36,47,25,0.010797,0.983966,0.731478,0.023385,0.005932


In [22]:
default_regularized = run_regularized_feature_selection(
    default_fe,
    target_col='y_default',
    label='default',
    already_engineered=True,
    screening_C=0.05,
    final_alpha=0.0001,
)

default_regularized['final_metrics_table']
default_regularized['source_feature_summary']
default_regularized['encoded_coef_table']

,encoded_feature,coefficient,abs_coefficient,odds_ratio,source_feature
0,num__unrate,0.902081,0.902081,2.464727,unrate
1,num__gs10,-0.874090,0.874090,0.417242,gs10
2,num__cpi_yoy,-0.501491,0.501491,0.605627,cpi_yoy
3,num__spread_at_origination,0.384245,0.384245,1.468505,spread_at_origination
4,num__mortgage_insurance_percentage_mi_pct,0.351688,0.351688,1.421465,mortgage_insurance_percentage_mi_pct
5,num__number_of_borrowers,-0.328900,0.328900,0.719715,number_of_borrowers
6,num__mortgage_treasury_spread,0.307953,0.307953,1.360638,mortgage_treasury_spread
7,num__current_interest_rate,0.255636,0.255636,1.291283,current_interest_rate
8,num__original_interest_rate,0.255636,0.255636,1.291283,original_interest_rate
9,num__mortgage_insurance_cancellation_indicator,-0.181533,0.181533,0.833991,mortgage_insurance_cancellation_indicator


In [23]:
default_regularized['final_metrics_table']

,model,train_rows,test_rows,n_features_raw,n_features_encoded_all,n_selected_encoded_features,test_event_rate,roc_auc,pr_auc,log_loss,brier_score
0,default_final_after_l1_screening,1424249,360913,36,47,17,0.000798,0.798036,0.004624,0.006101,0.000796


In [26]:
regularized_metrics_compare_df = pd.concat([
    prepay_regularized['final_metrics_table'].assign(outcome='prepay'),
    default_regularized['final_metrics_table'].assign(outcome='default'),
], ignore_index=True)


regularized_metrics_compare_df


,model,train_rows,test_rows,n_features_raw,n_features_encoded_all,n_selected_encoded_features,test_event_rate,roc_auc,pr_auc,log_loss,brier_score,outcome
0,prepay_final_after_l1_screening,1430718,354444,36,47,25,0.010797,0.983966,0.731478,0.023385,0.005932,prepay
1,default_final_after_l1_screening,1424249,360913,36,47,17,0.000798,0.798036,0.004624,0.006101,0.000796,default
